# Direct Coinbase 1d Bronze Ingestion

Heavy notebook implementation for Coinbase daily OHLC Bronze ingestion.

This notebook supports two execution modes:

- `backfill`: explicit inclusive date range
- `incremental`: rolling lookback window ending on the latest completed UTC day

The target table must already exist. Run `00_platform_setup_catalog_schema.ipynb` first.

In [ ]:
import hashlib
import json
import time as time_module
import uuid
from datetime import date, datetime, time, timedelta, timezone

import requests
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DoubleType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("product_ids", "BTC-USD")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "3")
dbutils.widgets.text("run_id", str(uuid.uuid4()))

RAW_PRODUCT_IDS = dbutils.widgets.get("product_ids").strip()
MODE = dbutils.widgets.get("mode").strip().lower()
START_DATE_RAW = dbutils.widgets.get("start_date").strip()
END_DATE_RAW = dbutils.widgets.get("end_date").strip()
LOOKBACK_DAYS_RAW = dbutils.widgets.get("lookback_days").strip()
RUN_ID = dbutils.widgets.get("run_id").strip()

TARGET_TABLE = "market_macro.brz_market.raw_coinbase_ohlc_1d"
COINBASE_API_BASE_URL = "https://api.exchange.coinbase.com"
MAX_CANDLES_PER_REQUEST = 300
REQUEST_TIMEOUT_SECONDS = 15
MAX_RETRIES = 5
UTC = timezone.utc


In [ ]:
def parse_product_ids(raw_value: str) -> list[str]:
    product_ids = [item.strip().upper() for item in raw_value.split(",") if item.strip()]
    product_ids = list(dict.fromkeys(product_ids))
    if not product_ids:
        raise ValueError("product_ids cannot be empty")
    return product_ids


def parse_iso_date(field_name: str, raw_value: str) -> date:
    try:
        return datetime.strptime(raw_value, "%Y-%m-%d").date()
    except ValueError as exc:
        raise ValueError(f"{field_name} must be in YYYY-MM-DD format") from exc


def resolve_date_window(
    mode: str,
    start_date_raw: str,
    end_date_raw: str,
    lookback_days_raw: str,
) -> tuple[date, date]:
    latest_complete_date = datetime.now(UTC).date() - timedelta(days=1)

    if mode not in {"backfill", "incremental"}:
        raise ValueError("mode must be either 'backfill' or 'incremental'")

    if mode == "backfill":
        if not start_date_raw or not end_date_raw:
            raise ValueError("backfill mode requires both start_date and end_date")

        start_date = parse_iso_date("start_date", start_date_raw)
        end_date = parse_iso_date("end_date", end_date_raw)
    else:
        lookback_days = int(lookback_days_raw or "0")
        if lookback_days < 1:
            raise ValueError("lookback_days must be >= 1 in incremental mode")

        end_date = parse_iso_date("end_date", end_date_raw) if end_date_raw else latest_complete_date
        start_date = end_date - timedelta(days=lookback_days - 1)

    if start_date > end_date:
        raise ValueError("start_date cannot be after end_date")

    if end_date > latest_complete_date:
        raise ValueError(
            f"end_date must be <= latest completed UTC day: {latest_complete_date.isoformat()}"
        )

    return start_date, end_date


def build_request_windows(start_date: date, end_date: date) -> list[tuple[datetime, datetime]]:
    windows: list[tuple[datetime, datetime]] = []
    current_start = datetime.combine(start_date, time.min, tzinfo=UTC)
    end_exclusive = datetime.combine(end_date + timedelta(days=1), time.min, tzinfo=UTC)

    while current_start < end_exclusive:
        current_end = min(current_start + timedelta(days=MAX_CANDLES_PER_REQUEST), end_exclusive)
        windows.append((current_start, current_end))
        current_start = current_end

    return windows


def to_iso_z(value: datetime) -> str:
    return value.isoformat().replace("+00:00", "Z")


def request_coinbase_json(session: requests.Session, url: str, params: dict) -> list:
    headers = {
        "Accept": "application/json",
        "User-Agent": "market-macro-lakehouse/phase1",
    }

    for attempt in range(MAX_RETRIES):
        try:
            response = session.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)
            if response.status_code == 200:
                return response.json()

            if response.status_code == 429 or 500 <= response.status_code < 600:
                wait_seconds = (2 ** attempt) + 0.5
                print(
                    f"Retryable Coinbase response {response.status_code} for {url}; waiting {wait_seconds:.1f}s"
                )
                time_module.sleep(wait_seconds)
                continue

            raise RuntimeError(
                f"Coinbase API error {response.status_code} for {url}: {response.text}"
            )
        except requests.RequestException as exc:
            if attempt == MAX_RETRIES - 1:
                raise RuntimeError(f"Coinbase request failed for {url}") from exc

            wait_seconds = (2 ** attempt) + 0.5
            print(f"Request exception for {url}: {exc}; waiting {wait_seconds:.1f}s before retry")
            time_module.sleep(wait_seconds)

    raise RuntimeError(f"Exhausted retries for {url}")


def fetch_coinbase_candles_1d(
    session: requests.Session,
    product_id: str,
    start_date: date,
    end_date: date,
) -> tuple[list[dict], dict]:
    url = f"{COINBASE_API_BASE_URL}/products/{product_id}/candles"
    records: list[dict] = []
    stats = {
        "api_rows_fetched": 0,
        "rows_in_requested_range": 0,
        "request_windows": 0,
    }

    for window_start, window_end in build_request_windows(start_date, end_date):
        params = {
            "start": to_iso_z(window_start),
            "end": to_iso_z(window_end),
            "granularity": 86400,
        }

        payload = request_coinbase_json(session, url, params)
        stats["request_windows"] += 1
        stats["api_rows_fetched"] += len(payload)

        for candle in payload:
            if not isinstance(candle, list) or len(candle) != 6:
                continue

            timestamp_seconds, low, high, open_value, close_value, volume = candle
            bar_date = datetime.fromtimestamp(int(timestamp_seconds), tz=UTC).date()

            if bar_date < start_date or bar_date > end_date:
                continue

            payload_hash = hashlib.sha256(
                json.dumps(
                    {"product_id": product_id, "candle": candle},
                    separators=(",", ":"),
                ).encode("utf-8")
            ).hexdigest()

            stats["rows_in_requested_range"] += 1
            records.append(
                {
                    "product_id": product_id,
                    "bar_date": bar_date,
                    "open": float(open_value),
                    "high": float(high),
                    "low": float(low),
                    "close": float(close_value),
                    "volume": float(volume),
                    "source_window_start": window_start,
                    "source_window_end": window_end,
                    "ingested_at": datetime.now(UTC),
                    "run_id": RUN_ID,
                    "payload_hash": payload_hash,
                }
            )

        time_module.sleep(0.3)

    return records, stats


In [ ]:
def run_coinbase_bronze_ingestion() -> dict:
    product_ids = parse_product_ids(RAW_PRODUCT_IDS)
    start_date, end_date = resolve_date_window(MODE, START_DATE_RAW, END_DATE_RAW, LOOKBACK_DAYS_RAW)

    if not spark.catalog.tableExists(TARGET_TABLE):
        raise RuntimeError(
            f"Target table {TARGET_TABLE} does not exist. Run 00_platform_setup_catalog_schema.ipynb first."
        )

    schema = StructType([
        StructField("product_id", StringType(), False),
        StructField("bar_date", DateType(), False),
        StructField("open", DoubleType(), True),
        StructField("high", DoubleType(), True),
        StructField("low", DoubleType(), True),
        StructField("close", DoubleType(), True),
        StructField("volume", DoubleType(), True),
        StructField("source_window_start", TimestampType(), False),
        StructField("source_window_end", TimestampType(), False),
        StructField("ingested_at", TimestampType(), False),
        StructField("run_id", StringType(), False),
        StructField("payload_hash", StringType(), False),
    ])

    all_records: list[dict] = []
    per_product_stats: dict[str, dict] = {}
    session = requests.Session()

    try:
        for product_id in product_ids:
            print(
                f"Fetching {product_id} from {start_date.isoformat()} to {end_date.isoformat()} in {MODE} mode"
            )
            records, stats = fetch_coinbase_candles_1d(session, product_id, start_date, end_date)
            all_records.extend(records)
            per_product_stats[product_id] = stats
            print(
                f"{product_id}: fetched={stats['api_rows_fetched']} in_range={stats['rows_in_requested_range']} windows={stats['request_windows']}"
            )
    finally:
        session.close()

    rows_fetched = sum(stats["api_rows_fetched"] for stats in per_product_stats.values())
    rows_after_filter = len(all_records)

    if not all_records:
        return {
            "status": "success_empty",
            "mode": MODE,
            "product_ids": product_ids,
            "start_date": start_date.isoformat(),
            "end_date": end_date.isoformat(),
            "rows_fetched": rows_fetched,
            "rows_after_filter": 0,
            "rows_after_dedup": 0,
            "rows_to_update": 0,
            "rows_to_insert": 0,
            "rows_merged": 0,
            "run_id": RUN_ID,
            "target_table": TARGET_TABLE,
            "per_product_stats": per_product_stats,
        }

    raw_df = spark.createDataFrame(all_records, schema=schema)
    date_filtered_df = raw_df.filter(
        (F.col("bar_date") >= F.lit(start_date)) & (F.col("bar_date") <= F.lit(end_date))
    )

    dedup_window = Window.partitionBy("product_id", "bar_date").orderBy(
        F.col("source_window_end").desc(),
        F.col("ingested_at").desc(),
        F.col("payload_hash").desc(),
    )

    deduped_df = (
        date_filtered_df
        .withColumn("_row_number", F.row_number().over(dedup_window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )

    rows_after_dedup = deduped_df.count()
    existing_key_count = (
        deduped_df.select("product_id", "bar_date")
        .join(
            spark.table(TARGET_TABLE).select("product_id", "bar_date"),
            on=["product_id", "bar_date"],
            how="inner",
        )
        .count()
    )

    DeltaTable.forName(spark, TARGET_TABLE).alias("tgt").merge(
        deduped_df.alias("src"),
        "tgt.product_id = src.product_id AND tgt.bar_date = src.bar_date",
    ).whenMatchedUpdate(
        set={
            "open": "src.open",
            "high": "src.high",
            "low": "src.low",
            "close": "src.close",
            "volume": "src.volume",
            "source_window_start": "src.source_window_start",
            "source_window_end": "src.source_window_end",
            "ingested_at": "src.ingested_at",
            "run_id": "src.run_id",
            "payload_hash": "src.payload_hash",
        }
    ).whenNotMatchedInsertAll().execute()

    display(deduped_df.orderBy("product_id", "bar_date"))

    return {
        "status": "success",
        "mode": MODE,
        "product_ids": product_ids,
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "rows_fetched": rows_fetched,
        "rows_after_filter": rows_after_filter,
        "rows_after_dedup": rows_after_dedup,
        "rows_to_update": existing_key_count,
        "rows_to_insert": rows_after_dedup - existing_key_count,
        "rows_merged": rows_after_dedup,
        "run_id": RUN_ID,
        "target_table": TARGET_TABLE,
        "per_product_stats": per_product_stats,
    }


result = run_coinbase_bronze_ingestion()
result
